# Imports and MLflow Configuration

In [1]:
import pandas as pd
import numpy as np
import mlflow
import xgboost as xgb
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import roc_auc_score, precision_score, recall_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Import our custom transformer from the src module we created in Phase 3
import sys
sys.path.append('../')
from src.features import CreditRiskFeatureEngineer

# Set up MLflow tracking
mlflow.set_tracking_uri("sqlite:///mlflow.db") # Stores metadata in a local SQLite DB
mlflow.set_experiment("Credit_Risk_Scoring")

print("MLflow configured. Experiment tracking is active.")

ModuleNotFoundError: No module named 'pandas'

In [2]:
df = pd.read_csv('../data/cs-training.csv')
if 'Unnamed: 0' in df.columns:
    df = df.drop('Unnamed: 0', axis=1)

X = df.drop('SeriousDlqin2yrs', axis=1)
y = df['SeriousDlqin2yrs']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Rebuild the preprocessing pipeline
preprocessor = Pipeline(steps=[
    ('feature_engineer', CreditRiskFeatureEngineer()),
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Fit and transform training data; ONLY transform test data
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [4]:
# Formula: Number of negative instances / Number of positive instances
neg_class_count = (y_train == 0).sum()
pos_class_count = (y_train == 1).sum()
imbalance_weight = neg_class_count / pos_class_count

print(f"Negative Class (Good): {neg_class_count}")
print(f"Positive Class (Default): {pos_class_count}")
print(f"Scale Pos Weight: {imbalance_weight:.2f}")

# Initialize the baseline model
xgb_baseline = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='auc',
    scale_pos_weight=imbalance_weight,
    random_state=42,
    early_stopping_rounds=10 # Prevents overfitting by stopping if validation score stops improving
)

Negative Class (Good): 111979
Positive Class (Default): 8021
Scale Pos Weight: 13.96


In [5]:
param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [100, 200, 300],
    'subsample': [0.8, 1.0]
}

print("Starting hyperparameter search...")
# 3-Fold Cross Validation ensures our model generalizes well
search = RandomizedSearchCV(
    xgb_baseline, 
    param_distributions=param_grid, 
    n_iter=5, # Number of parameter settings that are sampled (keep low for initial testing)
    scoring='roc_auc', 
    cv=3, 
    random_state=42,
    n_jobs=-1 # Use all CPU cores
)

# Start an MLflow Run
with mlflow.start_run(run_name="XGBoost_RandomSearch"):
    # Note: We must pass eval_set to fit() because we enabled early_stopping_rounds
    search.fit(
        X_train_processed, 
        y_train,
        eval_set=[(X_test_processed, y_test)],
        verbose=False
    )
    
    best_model = search.best_estimator_
    best_params = search.best_params_
    
    # 1. Log the best parameters to MLflow
    mlflow.log_params(best_params)
    
    # 2. Evaluate on the test set
    y_pred = best_model.predict(X_test_processed)
    y_prob = best_model.predict_proba(X_test_processed)[:, 1]
    
    auc_score = roc_auc_score(y_test, y_prob)
    prec_score = precision_score(y_test, y_pred)
    rec_score = recall_score(y_test, y_pred)
    
    # 3. Log the metrics to MLflow
    mlflow.log_metric("roc_auc", auc_score)
    mlflow.log_metric("precision", prec_score)
    mlflow.log_metric("recall", rec_score)
    
    # 4. Log the model artifact itself
    mlflow.xgboost.log_model(best_model, "xgboost_model")
    
    print(f"Run completed successfully.")
    print(f"Best Params: {best_params}")
    print(f"Test ROC-AUC: {auc_score:.4f}")
    print(f"Test Precision: {prec_score:.4f}")
    print(f"Test Recall: {rec_score:.4f}")

Starting hyperparameter search...


2026/11/16 09:38:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Run completed successfully.
Best Params: {'subsample': 0.8, 'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.1}
Test ROC-AUC: 0.8695
Test Precision: 0.2169
Test Recall: 0.7791
